In [1]:
# STEP 1: SETUP & INSTALLATIONS
!pip install -q transformers moviepy scenedetect git+https://github.com/openai/whisper.git ffmpeg-python nltk fuzzywuzzy python-Levenshtein

import os
import tempfile
from moviepy.editor import VideoFileClip, concatenate_videoclips
import whisper
from scenedetect import VideoManager, SceneManager
from scenedetect.detectors import ContentDetector
from transformers import BartTokenizer, BartForConditionalGeneration
from nltk.tokenize import sent_tokenize
from fuzzywuzzy import fuzz
import nltk
nltk.download('punkt')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
# STEP 2: LOAD INPUT VIDEO
video_path = "/content/input/test.mp4"  # Upload your video here
assert os.path.exists(video_path), "Video not found! Please upload to /content/."
video_clip = VideoFileClip(video_path)

In [3]:
# STEP 3: SHOT DETECTION USING PYSCENEDETECT
video_manager = VideoManager([video_path])
scene_manager = SceneManager()
scene_manager.add_detector(ContentDetector(threshold=30.0))
video_manager.set_downscale_factor()
video_manager.start()
scene_manager.detect_scenes(frame_source=video_manager)
scene_list = scene_manager.get_scene_list()
print(f"Detected {len(scene_list)} scenes.")

ERROR:pyscenedetect:VideoManager is deprecated and will be removed.
INFO:pyscenedetect:Loaded 1 video, framerate: 25.000 FPS, resolution: 640 x 360
INFO:pyscenedetect:Detecting scenes...


Detected 99 scenes.


In [4]:
# STEP 4: DEMUX AUDIO FROM EACH SHOT - WILL GIVE THE TRANSCRIPT FILE.
shots = []
asr_model = whisper.load_model("medium")

for i, (start, end) in enumerate(scene_list):
    shot_clip = video_clip.subclip(start.get_seconds(), end.get_seconds())
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        audio_path = tmp.name
    shot_clip.audio.write_audiofile(audio_path, fps=16000, nbytes=2, codec="pcm_s16le", verbose=False, logger=None)

    result = asr_model.transcribe(
        audio_path,
        language="en",
        word_timestamps=True,
        condition_on_previous_text=False
    )

    transcript = result['text'].strip()
    os.remove(audio_path)

    skip_phrases = ["thank you", "thanks for watching", "you", ""]
    if transcript.lower() in skip_phrases:
        continue

    shots.append({
        "index": i,
        "start_time": start.get_seconds(),
        "end_time": end.get_seconds(),
        "text": transcript,
        "clip": shot_clip
    })

print("✅ Audio transcription complete for all shots.")

# Transcription to /content/TRANSCRIBE.TXT
with open("/content/TRANSCRIBE.TXT", "w", encoding="utf-8") as f:
    for shot in shots:
        f.write(f"[{shot['start_time']} – {shot['end_time']}]:\n{shot['text']}\n\n")

print("Transcriptions saved to /content/TRANSCRIBE.TXT")


100%|██████████████████████████████████████| 1.42G/1.42G [00:13<00:00, 115MiB/s]


✅ Audio transcription complete for all shots.
Transcriptions saved to /content/TRANSCRIBE.TXT


In [5]:
# 📘 STEP 5: BART SUMMARIZATION WITH SMART FRAMING

from transformers import pipeline
import torch

# 🔄 Load summarizer
device = 0 if torch.cuda.is_available() else -1
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    tokenizer="facebook/bart-large-cnn",
    device=device
)

def frame_summary(summary):
    explanation_keywords = ['explain', 'demonstrate', 'describe', 'teach', 'how to', 'talk about', 'share', 'discuss']
    if any(kw in summary.lower() for kw in explanation_keywords):
        if not summary.lower().startswith("the speaker"):
            return f"The speaker explains that {summary[0].lower() + summary[1:]}"
    return summary

with open("/content/SUMMARIZED_TEXT.TXT", "w", encoding="utf-8") as f:
    for shot in shots:
        text = shot['text'].strip()
        if len(text.split()) < 25:
            summary = text
        else:

            draft = summarizer(
                text,
                max_length=60,
                min_length=20,
                length_penalty=2.5,
                no_repeat_ngram_size=3,
                do_sample=False
            )[0]['summary_text']


            summary = summarizer(
                draft,
                max_length=20,
                min_length=5,
                length_penalty=3.0,
                no_repeat_ngram_size=3,
                do_sample=False
            )[0]['summary_text']

        # Apply conditional framing
        final_summary = frame_summary(summary.strip())
        shot['summary'] = final_summary
        f.write(f"[{shot['start_time']}–{shot['end_time']}]: {final_summary}\n")

print("✅ STEP 5 complete: Smart summaries saved to /content/SUMMARIZED_TEXT.TXT")


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Your max_length is set to 60, but your input_length is only 41. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=20)
Your max_length is set to 60, but your input_length is only 38. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)
Your max_length is set to 60, but your input_length is only 42. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=21)
Your max_length is set to 60, but your input_length is only 33. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...'

✅ STEP 5 complete: Smart summaries saved to /content/SUMMARIZED_TEXT.TXT


In [6]:
from fuzzywuzzy import fuzz

# —— STEP 5.5: SCORE EACH SHOT FOR RELEVANCE ——
for shot in shots:
    summary   = shot.get("summary", "")
    transcript = shot.get("text", "")
    score      = fuzz.partial_ratio(summary.lower(), transcript.lower())
    shot["score"] = score



# —— Insert this before your current Step 6 ——

# Total video length in seconds
video_duration = video_clip.duration

# Divide shots into three chronological parts
third = video_duration / 3
part1 = [s for s in shots if s["start_time"] <= third]
part2 = [s for s in shots if third < s["start_time"] <= 2 * third]
part3 = [s for s in shots if s["start_time"] > 2 * third]



In [7]:
# —— Revised STEP 6 Selection Loop ——

target_duration = 0.60 * video_duration
print(f"🎯 Target summary length: {target_duration:.2f} sec "
      f"(60% of full video: {video_duration:.2f} sec)\n")

# Sort each third by descending score
part1 = sorted(part1, key=lambda s: s['score'], reverse=True)
part2 = sorted(part2, key=lambda s: s['score'], reverse=True)
part3 = sorted(part3, key=lambda s: s['score'], reverse=True)

important_shots = []
dur = 0
i = 0

# Keep adding top shots round‑robin until we meet or exceed the target
while dur < target_duration and (i < len(part1) or i < len(part2) or i < len(part3)):
    for part in (part1, part2, part3):
        if i < len(part) and dur < target_duration:
            shot = part[i]
            shot_len = shot['end_time'] - shot['start_time']
            # Always add the shot, even if it pushes us slightly over
            important_shots.append(shot)
            dur += shot_len
    i += 1

# Chronological order
important_shots.sort(key=lambda s: s['start_time'])

print(f"\n🎯 Selected {len(important_shots)} shots with total duration {dur:.2f} sec "
      f"({(dur/video_duration)*100:.1f}% of original)\n")


🎯 Target summary length: 358.81 sec (60% of full video: 598.01 sec)


🎯 Selected 70 shots with total duration 360.68 sec (60.3% of original)



In [8]:
from moviepy.editor import concatenate_videoclips

# STEP 7
total_duration = sum([s['end_time'] - s['start_time'] for s in shots])
selected_duration = sum([s['end_time'] - s['start_time'] for s in important_shots])

# Sort shots by order of appearance
important_shots.sort(key=lambda s: s['start_time'])


if selected_duration > 0.6 * total_duration:
    reduced = []
    dur = 0
    for shot in important_shots:
        reduced.append(shot)
        dur += shot['end_time'] - shot['start_time']
        if dur >= 0.5 * total_duration:
            break
    important_shots = reduced

final_summary_duration = sum([s['end_time'] - s['start_time'] for s in important_shots])
print(f"🎯 Final summary video will be {final_summary_duration:.2f} seconds long "
      f"({(final_summary_duration / total_duration) * 100:.1f}% of original)")

# STEP 8: STITCH CLIPS TO FINAL VIDEO
final_clips = [s['clip'] for s in important_shots]
summary_video = concatenate_videoclips(final_clips, method="compose")

# Export
output_path = "/content/summary_video.mp4"
summary_video.write_videofile(output_path, codec="libx264", audio_codec="aac")

print(f"✅ Summary video saved to: {output_path}")


🎯 Final summary video will be 295.00 seconds long (50.4% of original)
Moviepy - Building video /content/summary_video.mp4.
MoviePy - Writing audio in summary_videoTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/summary_video.mp4



Moviepy - Done !
Moviepy - video ready /content/summary_video.mp4
✅ Summary video saved to: /content/summary_video.mp4


#Text summarization

In [12]:
# --- robust summarization cell (paste after you have `transcript` defined) ---
import os
import gc
import torch
from transformers import AutoTokenizer, pipeline

# CONFIG
SUM_MODEL = "sshleifer/distilbart-cnn-12-6"
# safe max tokens input to the model (model limit is 1024 -> use margin)
MAX_INPUT_TOKENS = 900
# output file path (ensure this matches your earlier variable)
TXT_OUT = globals().get("TXT_OUT", "/content/output/summary.txt")
os.makedirs(os.path.dirname(TXT_OUT), exist_ok=True)

# Helper: free GPU memory
def free_gpu():
    try:
        torch.cuda.empty_cache()
        gc.collect()
    except Exception:
        pass

# Try to create a pipeline with fallbacks: normal GPU -> low-memory float16 -> CPU
def make_summarizer(model_name, device_idx):
    # 1) try default pipeline (lets Transformers handle device placement)
    try:
        print(f"Attempting pipeline(model={model_name}, device={device_idx}) ...")
        summ = pipeline("summarization", model=model_name, device=device_idx)
        print("Pipeline created (default).")
        return summ
    except Exception as e:
        print("Default pipeline failed:", repr(e))

    # 2) try lower-memory attempt (float16, low_cpu_mem_usage) if GPU requested
    if device_idx >= 0 and torch.cuda.is_available():
        try:
            print("Retrying pipeline with low_cpu_mem_usage and float16 (if supported)...")
            # model_kwargs is passed through to from_pretrained in many transformers versions
            summ = pipeline(
                "summarization",
                model=model_name,
                tokenizer=model_name,
                device=device_idx,
                framework="pt",
                # model_kwargs here helps some transformers versions reduce CPU memory and use fp16 on load
                model_kwargs={"torch_dtype": torch.float16, "low_cpu_mem_usage": True}
            )
            print("Pipeline created (float16 / low_cpu_mem_usage).")
            return summ
        except Exception as e2:
            print("Float16/low-memory attempt failed:", repr(e2))
            print("Falling back to CPU pipeline...")

    # 3) CPU fallback
    free_gpu()
    print("Creating CPU pipeline (device=-1). This will be slower but is robust.")
    summ = pipeline("summarization", model=model_name, device=-1)
    return summ

# determine device index for pipeline() argument
device_idx = 0 if torch.cuda.is_available() else -1
print("torch.cuda.is_available():", torch.cuda.is_available(), " -> device_idx =", device_idx)

# load tokenizer (needed for token-aware chunking)
print("Loading tokenizer (lightweight) ...")
tokenizer = AutoTokenizer.from_pretrained(SUM_MODEL)

# build summarizer with fallbacks
summarizer = make_summarizer(SUM_MODEL, device_idx)

# token-aware chunking function
def chunk_by_tokens(text, tokenizer, max_tokens=MAX_INPUT_TOKENS):
    """Return list of decoded text chunks whose token length <= max_tokens (approx)."""
    if not text:
        return []
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, len(token_ids), max_tokens):
        chunk_ids = token_ids[i : i + max_tokens]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        chunks.append(chunk_text)
    return chunks

# Split transcript into safe token chunks
print("Chunking transcript by tokens ...")
token_chunks = chunk_by_tokens(transcript, tokenizer, max_tokens=MAX_INPUT_TOKENS)
print(f"Transcript split into {len(token_chunks)} token-aware chunk(s).")

# Summarize each chunk robustly. If GPU errors happen mid-loop, fall back to CPU and resume.
chunk_summaries = []
use_cpu_fallback = False

for idx, chunk in enumerate(token_chunks):
    try:
        out = summarizer(
            chunk,
            max_length=100,
            min_length=20,
            do_sample=False,
            truncation=True
        )
        summary_text = out[0]["summary_text"].strip()
        chunk_summaries.append(summary_text)
        print(f"Chunk {idx+1}/{len(token_chunks)} summarized (summary len {len(summary_text)}).")
    except Exception as e:
        print(f"Error summarizing chunk {idx+1}: {repr(e)}")
        if not use_cpu_fallback:
            # recreate summarizer on CPU and resume
            print("Attempting CPU fallback for summarization (recreating pipeline on CPU)...")
            summarizer = make_summarizer(SUM_MODEL, device_idx=-1)
            use_cpu_fallback = True
            try:
                out = summarizer(
                    chunk,
                    max_length=100,
                    min_length=20,
                    do_sample=False,
                    truncation=True
                )
                summary_text = out[0]["summary_text"].strip()
                chunk_summaries.append(summary_text)
                print(f"Chunk {idx+1} summarized on CPU fallback.")
                continue
            except Exception as e2:
                print("CPU fallback also failed:", repr(e2))
        # If we get here, summarization for this chunk failed — append a placeholder and continue
        chunk_summaries.append("[ERROR: chunk not summarized]")
        print(f"Skipping chunk {idx+1} (added placeholder).")

# Create single-sentence overview by summarizing the chunk summaries (two-stage summarization)
if len(chunk_summaries) == 0:
    overview = ""
elif len(chunk_summaries) == 1:
    overview = chunk_summaries[0]
else:
    joined = " ".join(chunk_summaries)
    try:
        ov_out = summarizer(joined, max_length=50, min_length=10, do_sample=False, truncation=True)
        overview = ov_out[0]["summary_text"].strip()
    except Exception as e:
        print("Overview summarization failed, falling back to first chunk summary. Error:", repr(e))
        overview = chunk_summaries[0]

# Compose final summary text
lines = ["=== VIDEO SUMMARY ===", overview, "", "Below are the important points from the video:"]
lines += [f"{i+1}. {pt}" for i, pt in enumerate(chunk_summaries)]
full_summary = "\n".join(lines)

# Save to disk
with open(TXT_OUT, "w", encoding="utf-8") as f:
    f.write(full_summary)

print("\n✅ Summary complete.")
print(f"Overview (short): {overview[:200]}{'...' if len(overview) > 200 else ''}")
print(f"Full summary written to: {TXT_OUT}")


torch.cuda.is_available(): True  -> device_idx = 0
Loading tokenizer (lightweight) ...
Attempting pipeline(model=sshleifer/distilbart-cnn-12-6, device=0) ...


Device set to use cuda:0


Default pipeline failed: AcceleratorError('CUDA error: device-side assert triggered\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n')
Retrying pipeline with low_cpu_mem_usage and float16 (if supported)...


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


Float16/low-memory attempt failed: AcceleratorError('CUDA error: device-side assert triggered\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n')
Falling back to CPU pipeline...
Creating CPU pipeline (device=-1). This will be slower but is robust.


Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (1038 > 1024). Running this sequence through the model will result in indexing errors


Chunking transcript by tokens ...
Transcript split into 2 token-aware chunk(s).
Chunk 1/2 summarized (summary len 283).
Chunk 2/2 summarized (summary len 207).

✅ Summary complete.
Overview (short): The human voice is the most powerful sound in the world, probably . It's the only one that can start a war or say I love you . But many people have the experience that when they speak people don't lis...
Full summary written to: /content/output/summary.txt
